## 1. Motivation and model overview {#motivation-and-model-overview}

A player in possession decides whether to pass and **where to direct the ball**. Pass selection models this decision as a probability distribution over potential destinations, conditional on attempting a pass. *Pass probability* estimates whether a pass to a specified destination will be completed. A destination can be attractive without being safe, and a safe destination need not be frequently selected. These components describe different aspects of the same decision.

The distinction is central to spatial models of football. Spearman's *Beyond Expected Goals* [@spearman2018beyond] places a pass selection model alongside control and scoring opportunity in a framework for evaluating off-ball attacking opportunities. Fernández, Bornn and Cervone [@fernandez2021] explicitly model pass selection as a normalized spatial probability surface within an expected-possession-value framework. PRESS follows this probabilistic interpretation of destination choice, while learning the utility of each destination using gradient-boosted trees instead of a convolutional network.

At each observed passing decision, the pitch is discretized into candidate destinations on a **one-metre grid**. The observed endpoint is mapped to its nearest cell centre. The resulting probabilities sum to one across the complete candidate set; they are conditional on a pass being attempted, irrespective of whether the observed pass was successful.

Context-label meanings and the previous action convention are defined in [Possessions and checkpoints](../setup/possession_and_checkpoint_setup.ipynb#context-labels).


## 2. Probabilistic destination-choice model {#probabilistic-destination-choice-model}

Let $s_i$ denote the tracked game state at passing decision $i$, and let $\mathcal G_i$ be its set of candidate destination cells. Each cell $j$ is represented by features $z_{ij}$ describing the pass and the current spatial configuration. A **single shared function** assigns a utility to each destination:

$$
u_{ij}=f_\psi(z_{ij}).$$

PRESS represents $f_\psi$ as an additive ensemble of XGBoost regression trees. Utilities express relative preference among destinations. The multinomial-logit model converts them into a categorical distribution:

$$
\boxed{\pi_{ij}=P(Y_i=j\mid s_i,\text{pass})=
\frac{\exp(u_{ij})}{\sum_{k\in\mathcal G_i}\exp(u_{ik})}.}
$$

The normalization couples every candidate in the same decision: changing the utility of one destination changes the probabilities of the others.

The ensemble is estimated from observed destinations by minimizing the categorical negative log-likelihood,

$$\mathcal L(\psi)=-\sum_i\log \pi_{i,y_i}(\psi),$$

where $y_i$ is the observed destination cell. The [benchmark](../../benchmarks/03_pass_selection.qmd) describes the boosting procedure, regularization, model selection and validation.

After fitting, a scalar temperature $T$ calibrates the predicted distribution,

$$\pi_{ij}^{(T)}=\frac{\pi_{ij}^{1/T}}{\sum_k \pi_{ik}^{1/T}},$$

without changing the ordering of destinations. The installed model uses the flat multinomial-logit specification; an exploratory cross-nested extension is evaluated separately in the benchmark.

## 3. Model features {#model-features}

The model uses **21 conceptual features: 19 numerical and two categorical**, in four groups. Each group answers one question about the decision. Each feature is described first in plain terms and then by how it is computed, and uses only information available at the passing decision; the realized receiver, pass outcome and subsequent events are excluded. Categorical variables are one-hot encoded using a vocabulary learned during fitting.

| Group | Question | Count |
|---|---|---:|
| Carrier state and ball retention | How fast is the carrier moving, and how risky is continuing the carry? | 2 |
| Spatial and tactical configuration | What is the current location's value and team structure? | 6 |
| Passing opportunities | What are the geometry, accessibility and tactical properties of each candidate destination? | 9 |
| Possession and match context | How did control start, and what is the match situation? | 4 |

### Carrier state and ball retention (2 features)

These features describe how the carrier is moving and how risky it would be to keep the ball. They are shared by all destinations in the same decision.

- **Carrier speed.** How fast the carrier is moving: running with the ball, or almost standing. It is the magnitude of the carrier's current velocity, $\lVert v_{\mathrm{bc}}\rVert$, in m/s.
- **Carry-loss probability $d(s)$.** The chance that the carrier loses the ball within one second if the carry continues. A high value means that keeping the ball is risky in the current situation. It is the probability of loss during a canonical one-second carry, from the [ball-drive risk model](../01_ball_drive_risk/01_ball_drive_risk.ipynb). Pass selection uses $d(s)$; action selection uses its complement $1-d(s)$, an alternative encoding of the same execution-risk information.

### Spatial and tactical configuration (6 features)

These features describe the value of the carrier's location and how the teams are arranged around it. They are evaluated at the carrier's **current position** and shared by all candidate destinations.

- **Own-team xT** and **opponent xT (2 features).** How valuable it is to hold the ball at the carrier's location: the first value is for the carrier's team and the second for the opponent, each in its own attacking orientation. The construction follows Singh [@singh2018xt] and is described in the [xT vignette](../spatial_features/expected_threat/11_expected_threat.ipynb).
- **Nearest own-team line index** and **nearest opposition line index (2 features).** Where the carrier is located relative to the pressure structure of each team. The index is the closest of three longitudinal line centres, following the spatial line representation of Fernández, Bornn and Cervone [@fernandez2021]. The lines use current player positions, with the carrier excluded from their own team's construction and the extreme-x goalkeeper proxy removed from each roster. Opposition indices follow the carrier team's attacking axis; own-team indices use the opposite-axis ordering.
- **Teammates ahead** and **opponents ahead (2 features).** How many players of each team are further toward the goal the carrier team attacks. They are counts strictly beyond the transverse line through the carrier, along the carrier team's attacking direction, and both rosters use the same direction. The carrier is excluded, tracked goalkeepers are retained, and players exactly on the line are excluded.

### Passing opportunities (9 features)

These features describe each candidate destination $j$, evaluated from the current frame, with its own geometry and tactical descriptors.

- **Pass distance.** How long the pass would be. It is the Euclidean distance from the carrier to the candidate destination, in m.
- **Pass angle.** Whether the pass follows or departs from the direction in which the carrier is already moving. It is the signed angle from the carrier's velocity to the carrier-to-destination vector, and is undefined for a stationary carrier.
- **Raw target pitch control $C_{\mathrm{att}}$.** How likely the attacking team is to control that point. It is the attacking team's control probability at the destination, excluding the carrier. It uses raw pitch control, without isotonic calibration; [pass probability](../02_pass_probability/02_pass_probability.ipynb) is the calibrated quantity used for execution-risk summaries in action selection.
- **Own-team xT** and **opponent xT at the destination (2 features).** How valuable it would be to hold the ball at the destination for each team, in its own attacking orientation.
- **Nearest own-team line index** and **nearest opposition line index at the destination (2 features).** Where the destination is located relative to the pressure structure of each team. They use the same line definitions and ordering as at the carrier, evaluated at the candidate point.
- **Teammates ahead** and **opponents ahead of the destination (2 features).** How many players of each team are further toward the attacked goal than the destination. They use the same strict longitudinal counts as at the carrier, with current player positions. A backward candidate may have more players ahead, since these are counts beyond each line.

For a player group $G$, query point $r$ and carrier-team attacking direction $\delta\in\{-1,+1\}$, the shared count convention is

$$
N_G(r)=\sum_{k\in G}\mathbf 1\{\delta\,x_k>\delta\,r_x\}.
$$

### Possession and match context (4 features)

These features describe how the possession began and the state of the match.

- **Start type (categorical).** How the player got the ball: the event through which the current individual possession began.
- **Previous action (categorical).** What the player did before: the previous canonical action. The start context is retained until a full one-second carry has been completed, after which it is `ball_drive`.
- **Match minute.** How far the match has advanced: the time elapsed in the match at the passing decision.
- **Score difference.** Whether the carrier's team is ahead or behind: carrier-team goals minus opponent goals known at that frame.

These definitions use the general event meanings and segmentation conventions documented in [Possessions and checkpoints](../setup/possession_and_checkpoint_setup.ipynb#context-labels). The same categories are used by action selection, with player identity and role left out.


## 4. Feature importance {#feature-importance}

The following results summarize **training split-gain importance** for the fitted full-data model, together with each feature's rank across the five held-out evaluation fits. Gain is an attribution of the ensemble's split improvements.

| Feature | Full-data gain share | Rank across five folds |
|---|---:|---:|
| Pass distance | 34.44% | 1–1 |
| Raw target pitch control | 20.91% | 2–2 |
| Pass angle | 14.41% | 3–3 |
| Teammates beyond target | 5.72% | 4–4 |
| Opponents beyond target | 5.04% | 5–7 |
| Carrier speed | 3.49% | 5–6 |

Distance, pitch control and angle account for **69.76%** of full-data split gain (rounded component shares) and occupy the same first three positions in every fold. This is consistent with geometry and destination accessibility being particularly informative for observed pass choice.

![Training split-gain allocation for the fitted model and five evaluation folds.](../../figures/feature_importance_pass_selection.png)

## 5. Application to a game state {#application-to-a-game-state}

Brisbane Roar FC–Perth Glory: N. Acosta chooses and completes a pass to J. Hingert (checkpoint 54295). Set `SKILLCORNER_OPENDATA` to your local open-data repository ([Data setup](../../data.qmd)). The dashed line shows the matched destination. Colors display relative preference: 100% is the most probable destination cell. The stored cell probabilities remain a distribution summing to one.

The action labels are retrieved with `get_frame_context`; the match minute is calculated from the timestamp returned by `get_frame`. The carrier-team score difference is read from the current possession's dynamical event. It is supplied explicitly because these processed tracking rows do not contain score information.


In [7]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
import jax

jax.config.update("jax_enable_x64", True)
from PRESS import PRESS

opendata = Path(os.environ.get("SKILLCORNER_OPENDATA", Path.cwd().parent / "opendata"))
match_dir = opendata / "data/matches/1925299"
press = PRESS.from_skillcorner(match_dir)
frame_id = 54295

from PRESS.model import PassSelectionModel
from PRESS.visualization import plot_pass_map

choice = PassSelectionModel.pretrained()
context = press.get_frame_context(frame_id)
frame = press.get_frame(frame_id).iloc[0]

# Match minute comes from the observed tracking timestamp.
match_clock_min = pd.to_timedelta(frame.timestamp).total_seconds() / 60

# The context getter supplies action labels, not the score. Read the known
# score from the dynamical event of the current individual possession.
possessions = press.get_player_possessions()
current_possession = possessions.loc[
    possessions.frame_start.le(frame_id) & possessions.frame_end.ge(frame_id)
].iloc[0]
events = pd.read_csv(
    opendata / "data/matches/1925299/1925299_dynamic_events.csv", low_memory=False,
)
origin = events.loc[
    events.event_type.eq("player_possession")
    & events.event_id.eq(current_possession.possession_id)
].iloc[0]
score_difference = float(origin.team_score - origin.opponent_team_score)

press.compute_ppcf(frame_id)
press.compute_ball_drive_risk(frame_id)
press.compute_pass_selection(
    frame_id, choice.xt_grid,
    start_type_raw=context["start_type"], previous_action_raw=context["previous_action"],
    match_clock_min=match_clock_min, score_difference_prefix=score_difference,
)
plot_pass_map(press, frame_id, field="pass_selection", pass_target=(-12.15, -28.05))
plt.savefig("figures/vignette_03_pass_selection_pitch.png", dpi=170)
plt.savefig("figures/vignette_03_pass_selection_pitch.svg")
plt.close()


![Pass selection on the real checkpoint; color intensity is relative to the most probable cell.](../../figures/vignette_03_pass_selection_pitch.png)

## 6. Using the fitted model {#using-the-fitted-model}

The model can also be loaded directly, with its saved feature vocabulary and fitted physical-model dependencies:

```python
from PRESS.model import PassSelectionModel
choice = PassSelectionModel.pretrained()
press = PRESS(tracking, pass_selection_model=choice)
press.compute_ppcf(frame_id)
press.compute_ball_drive_risk(frame_id, t_press=1.0)
probability = press.compute_pass_selection(
    frame_id, xT_grid,
    start_type_raw=context["start_type"], previous_action_raw=context["previous_action"],
    match_clock_min=match_clock_min, score_difference_prefix=score_difference,
)
```

Here `context`, `match_clock_min` and `score_difference` are the observed values recovered in the preceding example; retrieve them again when selecting another frame. The positional `xT_grid` argument remains for API compatibility; the installed choice model retains its fitted xT representation. The [benchmark](../../benchmarks/03_pass_selection.qmd) reports the evaluation protocol, training details, exploratory nested alternative and comparison with Fernández et al. [@fernandez2021].